# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1, 2, 3] [0, 1, 1] Обычное деление / делит каждый элемент пополам а // делит на 4 и отбрасывает дробный остаток:
2. [4, 4, 4] создает таблицу из единиц 3 строки по 4 единицы в каждой, идет вправо по каждой строке и складывает 4 единицы
3. горизонтальная строка [0, 1, 2] + вертикальный столбец [[0], [1], [2]] будет примерно 
[0 1 2]
[1 2 3]
[2 3 4]
4. 6, берет квадратный корень из каждого числа в [1, 4, 9] затем суммирует
5. 7.5, [1, 2, 3, 4] Возведение в квадрат ** 2 превращает их в [1, 4, 9, 16]. Метод .mean() находит среднее арифметическое

In [25]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [26]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [27]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [28]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [29]:
# 1
rev_sep = amount[clean & (month == "2026-09")].sum()
rev_oct = amount[clean & (month == "2026-10")].sum()
change_pct = (rev_oct / rev_sep - 1) * 100

print("Строк в clean:", clean.sum())
print("Сентябрь:", round(rev_sep))
print("Октябрь:", round(rev_oct))
print("Изменение (%):", round(change_pct, 2))
print("Вывод: В чистых данных", clean.sum(), "заказов, выручка в октябре изменилась на", round(change_pct, 2), "% (совпадает с расчетом ДЗ 2).")

Строк в clean: 4739
Сентябрь: 35243410
Октябрь: 34419520
Изменение (%): -2.34
Вывод: В чистых данных 4739 заказов, выручка в октябре изменилась на -2.34 % (совпадает с расчетом ДЗ 2).


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [30]:
# 2
cats = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

cat_table = np.array([[amount[clean & (cat == k) & (month == m)].sum() for m in months] for k in cats])

cat_totals = cat_table.sum(axis=1)
cat_change = (cat_table[:, 1] / cat_table[:, 0] - 1) * 100

for k, r in zip(cats, cat_table):
    print(k, round(r[0]), round(r[1]))

best_idx = cat_totals.argmax()
worst_idx = cat_change.argmin()

print("Больше всего выручки:", cats[best_idx], round(cat_totals[best_idx]))
print("Сильнее всех упала (%):", cats[worst_idx], round(cat_change[worst_idx], 2))
print("Вывод: Лидер по выручке за два месяца -", cats[best_idx], "(", round(cat_totals[best_idx]), "), а сильнее всех просела категория", cats[worst_idx], "(", round(cat_change[worst_idx], 2), "%).")

Дом 4997730 5125790
Книги 1065300 1140910
Красота 2610950 2678990
Одежда 9167100 8505730
Электроника 17402330 16968100
Больше всего выручки: Электроника 34370430
Сильнее всех упала (%): Одежда -7.21
Вывод: Лидер по выручке за два месяца - Электроника ( 34370430 ), а сильнее всех просела категория Одежда ( -7.21 %).


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [31]:
# 3
cat_counts = np.array([[(clean & (cat == k) & (month == m)).sum() for m in months] for k in cats])

avg_check = cat_table / cat_counts

for k, avg in zip(cats, avg_check):
    print(k, "средний чек в октябре:", round(avg[1]))

best_avg_idx = avg_check[:, 1].argmax()
print("Вывод: Самый высокий средний чек в октябре у категории", cats[best_avg_idx], "и составляет", round(avg_check[best_avg_idx, 1]))

Дом средний чек в октябре: 10768
Книги средний чек в октябре: 3816
Красота средний чек в октябре: 6159
Одежда средний чек в октябре: 13046
Электроника средний чек в октябре: 33600
Вывод: Самый высокий средний чек в октябре у категории Электроника и составляет 33600


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [32]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months] for c in cities])

city_shares = table / table.sum(axis=0)

for c, sh in zip(cities, city_shares):
    print(c, round(sh[0] * 100, 2), "% ->", round(sh[1] * 100, 2), "%")

print("Сумма столбцов:", city_shares.sum(axis=0))
print("Вывод: Доля Шымкента снизилась с", round(city_shares[5, 0] * 100, 2), "% в сентябре до", round(city_shares[5, 1] * 100, 2), "% в октябре.")

Актобе 7.31 % -> 7.38 %
Алматы 41.15 % -> 42.54 %
Астана 25.63 % -> 25.71 %
Атырау 2.62 % -> 2.82 %
Караганда 10.35 % -> 9.8 %
Шымкент 12.93 % -> 11.75 %
Сумма столбцов: [1. 1.]
Вывод: Доля Шымкента снизилась с 12.93 % в сентябре до 11.75 % в октябре.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [33]:
# 5
change = table[:, 1] - table[:, 0]
contrib = (change / change.sum()) * 100

for c, ch, co in zip(cities, change, contrib):
    print(c, "изменение:", round(ch), "| вклад в падение:", round(co, 1), "%")

worst_city_idx = change.argmin()
print("Вывод: Город", cities[worst_city_idx], "даёт", round(contrib[worst_city_idx], 1), "% всего падения выручки, причину нужно искать в доставке и отменах этого города.")

Актобе изменение: -36540 | вклад в падение: 4.4 %
Алматы изменение: 141950 | вклад в падение: -17.2 %
Астана изменение: -183690 | вклад в падение: 22.3 %
Атырау изменение: 45870 | вклад в падение: -5.6 %
Караганда изменение: -276840 | вклад в падение: 33.6 %
Шымкент изменение: -514640 | вклад в падение: 62.5 %
Вывод: Город Шымкент даёт 62.5 % всего падения выручки, причину нужно искать в доставке и отменах этого города.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [34]:
# 6
# Почему падает: у table форма (6, 2), а у table.sum(axis=1) форма (6,).
# Размеры справа 2 и 6 не совпадают. Нужен параметр keepdims=True, чтобы сумма имела форму (6, 1).

share = table / table.sum(axis=1, keepdims=True)

for c, sh in zip(cities, share):
    print(c, round(sh[0] * 100, 1), "% |", round(sh[1] * 100, 1), "%")

print("Сумма по строкам:", share.sum(axis=1))
print("Вывод: С параметром keepdims=True код работает без ошибок, в Шымкенте доля сентября", round(share[5, 0] * 100, 1), "%, а октября", round(share[5, 1] * 100, 1), "%.")

Актобе 50.4 % | 49.6 %
Алматы 49.8 % | 50.2 %
Астана 50.5 % | 49.5 %
Атырау 48.8 % | 51.2 %
Караганда 52.0 % | 48.0 %
Шымкент 53.0 % | 47.0 %
Сумма по строкам: [1. 1. 1. 1. 1. 1.]
Вывод: С параметром keepdims=True код работает без ошибок, в Шымкенте доля сентября 53.0 %, а октября 47.0 %.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [35]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [36]:
# 8